# PARC2026 — 71 M3 runner preflight

Notebook 70 の `READY_FOR_RUNNER` を受けて、M3実学習を始める前に source pin / config / dataset provenance / 69c streaming bridge を再検証します。

このNotebookは **trainingを開始しません**。SmolVLA と OpenVLA-OFT の固定sourceをcheckoutし、training entryの存在まで確認して `READY_FOR_BATCH_PROBE` を作ります。次工程はA100上の小さなforward/backward batch probeです。


In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_preflight'
PIN = 'b64e7a2e7ec481e17d3de5a3c2aef5f18f53cfda'
URL = 'https://github.com/yu37330/py_AI.git'

if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('71 preflight code:', got, flush=True)

subprocess.run([
    'python', '-u', str(REPO / 'tools/colab/run_m3_runner_preflight.py')
], cwd=str(REPO), check=True)
print('=== 71 COMPLETE ===', flush=True)
print('Next: A100 per-model batch/forward-backward probes; M3 benchmark training has NOT started.', flush=True)
